# `klygo.files` (`io.py`): Structured data I/O and conversion

Read and write structured data by filename extension, and convert between supported formats.

Run this notebook from top to bottom. It creates only temporary example files.

In [ ]:
!pip install git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare sample files

Run this setup again to reset the examples. No Drive mount or manual upload is required.

In [1]:
from tempfile import TemporaryDirectory
from klygo import files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-files-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-files-xi0i571j


In [2]:
from contextlib import closing
files.save(root / "config.json", {"batch": 16, "classes": ["cat", "dog"]}, verbose=False)
files.save(root / "records.csv", [{"image": "cat.jpg", "class": "cat"}, {"image": "dog.jpg", "class": "dog"}], verbose=False)
files.save(root / "events.jsonl", [{"epoch": 1}, {"epoch": 2}], verbose=False)
files.save(root / "notes.txt", "one\ntwo\n", verbose=False)
print("Structured sample files ready:", root)

Structured sample files ready: C:\Users\Mine\AppData\Local\Temp\klygo-files-xi0i571j


## Jump to an API

[`load`](#load) · [`iter_load`](#iter_load) · [`save`](#save) · [`convert`](#convert)

**Supported suffixes:** `.json`, `.jsonl`, `.yaml`, `.yml`, `.toml`, `.csv`, `.txt`, `.log`, `.ini`, `.cfg`, `.properties`, `.env`, `.xml`, `.pkl`, `.pickle`. The last two use Pickle and must only be loaded from trusted sources. Format shapes differ; CSV values are strings, whereas JSON values retain JSON types.

<a id="load"></a>

### `files.load`

Load structured data from a file using its filename extension.

**Signature:** `files.load(path: 'PathInput', as_lines: 'bool' = False, verbose: 'bool' = True) -> 'Any'`

- `path`: Path to inspect or operate on.
- `as_lines`: Return TXT and LOG files as lines without newline characters.
- `verbose`: Display a progress indicator.

**Returns:** `Any` — Decoded content; the concrete type depends on the format.

The final filename extension chooses the parser. CSV gives a list of string-valued row dictionaries; JSONL gives a list of JSON values; text gives a string unless `as_lines=True`; YAML/TOML/INI/ENV/XML have format-specific shapes. Pickle can execute code, so load it only from a trusted source.

In [3]:
config = files.load(root / 'config.json', verbose=False)
rows = files.load(root / 'records.csv', verbose=False)
lines = files.load(root / 'notes.txt', as_lines=True, verbose=False)
print(config, rows, lines)

{'batch': 16, 'classes': ['cat', 'dog']} [{'image': 'cat.jpg', 'class': 'cat'}, {'image': 'dog.jpg', 'class': 'dog'}] ['one', 'two']


**Another useful pattern**

In [4]:
formats = ['config.json', 'records.csv', 'events.jsonl', 'notes.txt']
for filename in formats:
    value = files.load(root / filename, verbose=False)
    print(filename, type(value).__name__, value)

config.json dict {'batch': 16, 'classes': ['cat', 'dog']}
records.csv list [{'image': 'cat.jpg', 'class': 'cat'}, {'image': 'dog.jpg', 'class': 'dog'}]
events.jsonl list [{'epoch': 1}, {'epoch': 2}]
notes.txt str one
two



**Expected error and recovery**

In [5]:
unknown = root / "payload.unknown"
unknown.write_text("data", encoding="utf-8")
try:
    files.load(unknown, verbose=False)
except ValueError as error:
    print("ValueError:", error)

ValueError: Unsupported format: '.unknown', supported: ['.cfg', '.csv', '.env', '.ini', '.json', '.jsonl', '.log', '.pickle', '.pkl', '.properties', '.toml', '.txt', '.xml', '.yaml', '.yml']


<a id="iter_load"></a>

### `files.iter_load`

Yield individual JSONL values or CSV rows using UTF-8 decoding.

**Signature:** `files.iter_load(path: 'PathInput') -> 'Generator[Any, None, None]'`

- `path`: Existing JSONL or CSV file. Other formats are intentionally unsupported.

**Returns:** `generator[Any, None, None]` — JSON values or CSV row dictionaries, read one record at a time.

Only `.jsonl` and `.csv` are supported. Validation occurs on call; file opening and record parsing occur during iteration. Use `closing()` when stopping early so the open file is released promptly.

In [6]:
from contextlib import closing
with closing(files.iter_load(root / 'events.jsonl')) as records:
    for record in records:
        print(record)
with closing(files.iter_load(root / 'records.csv')) as records:
    print(next(records))

{'epoch': 1}
{'epoch': 2}
{'image': 'cat.jpg', 'class': 'cat'}


**Another useful pattern**

In [7]:
with closing(files.iter_load(root / 'events.jsonl')) as events:
    first_event = next(events)
    print('First event only:', first_event)
print('Full list:', files.load(root / 'events.jsonl', verbose=False))

First event only: {'epoch': 1}
Full list: [{'epoch': 1}, {'epoch': 2}]


**Expected error and recovery**

In [8]:
try:
    files.iter_load(root / "config.json")
except ValueError as error:
    print("ValueError:", error)

# Fix: use load() for JSON, or iter_load() for CSV and JSONL.
print(files.load(root / "config.json", verbose=False))

ValueError: iter_load() supports only .jsonl and .csv files
{'batch': 16, 'classes': ['cat', 'dog']}


<a id="save"></a>

### `files.save`

Save structured data using the destination filename extension.

**Signature:** `files.save(path: 'PathInput', data: 'Any', overwrite: 'bool' = False, verbose: 'bool' = True, indent: 'int' = 4, fieldnames: 'Optional[List[str]]' = None, *, atomic: 'bool' = True) -> 'None'`

- `path`: Path to inspect or operate on.
- `data`: Python object to serialize.
- `overwrite`: Allow an existing destination to be replaced.
- `verbose`: Display a progress indicator.
- `indent`: Number of spaces used to indent JSON output.
- `fieldnames`: Ordered CSV column names; inferred from the first row when omitted.
- `atomic`: Serialize to a sibling temporary file, then publish the completed file.

**Returns:** `None` — The destination is written as a side effect.

The final extension selects the serializer. Parents are created automatically. `overwrite=False` protects existing entries, while `atomic=True` writes a sibling temporary file before publishing; it does not guarantee power-loss durability. `indent` affects JSON; `fieldnames` controls CSV column order.

In [9]:
saved = root / 'output' / 'settings.yaml'
files.save(saved, {'batch': 16, 'augment': True}, verbose=False)
print(files.load(saved, verbose=False))
try:
    files.save(saved, {'batch': 32}, verbose=False)
except FileExistsError:
    print('Existing output was preserved; use overwrite=True intentionally.')

{'batch': 16, 'augment': True}
Existing output was preserved; use overwrite=True intentionally.


**Another useful pattern**

In [10]:
rows = [{'image': 'cat.jpg', 'score': 0.95}, {'image': 'dog.jpg', 'score': 0.91}]
csv_path = root / 'output' / 'scores.csv'
files.save(csv_path, rows, fieldnames=['image', 'score'], verbose=False)
print(files.load(csv_path, verbose=False))

[{'image': 'cat.jpg', 'score': '0.95'}, {'image': 'dog.jpg', 'score': '0.91'}]


**Expected error and recovery**

In [11]:
target = root / "output" / "settings.yaml"
try:
    files.save(target, {"batch": 64}, verbose=False)
except FileExistsError as error:
    print("FileExistsError:", error)

files.save(target, {"batch": 64}, overwrite=True, verbose=False)
print(files.load(target, verbose=False))

FileExistsError: File already exists: C:\Users\Mine\AppData\Local\Temp\klygo-files-xi0i571j\output\settings.yaml. Use overwrite=True to replace it.
{'batch': 64}


<a id="convert"></a>

### `files.convert`

Convert data between two supported structured-data formats.

**Signature:** `files.convert(source: 'PathInput', target: 'PathInput', overwrite: 'bool' = False, verbose: 'bool' = True) -> 'Path'`

- `source`: Source URL or filesystem path.
- `target`: Destination filesystem path.
- `overwrite`: Allow an existing destination to be replaced.
- `verbose`: Display a progress indicator.

**Returns:** `pathlib.Path` — Path to the converted file.

Conversion loads the source and saves the destination; it is not a streaming transform. The target format is determined by its extension. An existing target requires `overwrite=True`.

In [12]:
converted = files.convert(root / 'config.json', root / 'output' / 'config.yaml', verbose=False)
print(converted, files.load(converted, verbose=False))

C:\Users\Mine\AppData\Local\Temp\klygo-files-xi0i571j\output\config.yaml {'batch': 16, 'classes': ['cat', 'dog']}


**Another useful pattern**

In [13]:
csv_target = files.convert(root / 'records.csv', root / 'output' / 'records.json', verbose=False)
print('Converted rows:', files.load(csv_target, verbose=False))

Converted rows: [{'image': 'cat.jpg', 'class': 'cat'}, {'image': 'dog.jpg', 'class': 'dog'}]


**Expected error and recovery**

In [14]:
target = root / "output" / "config.yaml"
try:
    files.convert(root / "config.json", target, verbose=False)
except FileExistsError as error:
    print("FileExistsError:", error)

files.convert(root / "config.json", target, overwrite=True, verbose=False)
print(files.load(target, verbose=False))

FileExistsError: File already exists: C:\Users\Mine\AppData\Local\Temp\klygo-files-xi0i571j\output\config.yaml. Use overwrite=True to replace it.
{'batch': 16, 'classes': ['cat', 'dog']}


## Put it together

In [15]:
with closing(files.iter_load(root / 'records.csv')) as rows:
    cleaned = [{'image': row['image'], 'class': row['class']} for row in rows]
files.save(root / 'output' / 'index.json', cleaned, verbose=False)
print(files.load(root / 'output' / 'index.json', verbose=False))

[{'image': 'cat.jpg', 'class': 'cat'}, {'image': 'dog.jpg', 'class': 'dog'}]


## Remember

- `load()` materializes a complete object in memory; for large JSONL, CSV, or text streams, use `iter_load()` and close unfinished iterators.
- Next in `klygo.files`: continue with [Document_Files_Download.ipynb](Document_Files_Download.ipynb) ([Colab](https://colab.research.google.com/drive/1Cte1cGBeOPGVAotapky_XY6g2whV2fRw?usp=sharing)) or return to [Document_Files_Overview.ipynb](Document_Files_Overview.ipynb) ([Colab](https://colab.research.google.com/drive/12PhuA8SVe_Q8jfC4f5klsgQifR6Vdf4H?usp=sharing)).
- Related Klygo modules: [`klygo.archive`](../archive/Document_Archive_Overview.ipynb) · [`klygo.config`](../config/Document_Config_Overview.ipynb).